# AI3003 Lab2：CNN 识别 Fashion-MNIST

## 数据加载与实验准备


In [ ]:
import random
import sys
from pathlib import Path

import numpy as np
import torch

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

ROOT = Path.cwd()
if not (ROOT / "cnn.py").exists():
    ROOT = ROOT / "lab" / "lab2"

FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Using device: {device}")
print(f"Working directory: {ROOT.resolve()}")


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)),
])

full_train = datasets.FashionMNIST(
    root=ROOT / "data",
    train=True,
    download=True,
    transform=transform,
)
full_test = datasets.FashionMNIST(
    root=ROOT / "data",
    train=False,
    download=True,
    transform=transform,
)

train_size = int(len(full_train) * 5 / 6)
val_size = len(full_train) - train_size
split_generator = torch.Generator().manual_seed(seed)
train_set, val_set = random_split(full_train, [train_size, val_size], generator=split_generator)

BATCH_SIZE = 512
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE)
test_loader = DataLoader(full_test, batch_size=BATCH_SIZE)

class_names = full_train.classes
mean = torch.tensor(0.2860)
std = torch.tensor(0.3530)

print(f"训练集大小: {len(train_set)}")
print(f"验证集大小: {len(val_set)}")
print(f"测试集大小: {len(full_test)}")
print(f"类别: {class_names}")


In [ ]:
import matplotlib.pyplot as plt

samples_per_class = 3
class_samples = {i: [] for i in range(len(class_names))}

for image, label in full_train:
    if len(class_samples[label]) < samples_per_class:
        class_samples[label].append(image)
    if all(len(images) == samples_per_class for images in class_samples.values()):
        break

fig, axes = plt.subplots(samples_per_class, len(class_names), figsize=(14, 4.5))
fig.suptitle("Fashion-MNIST 示例图像", fontsize=16, y=1.02)

for row in range(samples_per_class):
    for col, class_name in enumerate(class_names):
        ax = axes[row, col]
        image = class_samples[col][row] * std + mean
        ax.imshow(image.squeeze(), cmap="gray")
        ax.axis("off")
        if row == 0:
            ax.set_title(class_name, fontsize=9)

plt.tight_layout()
plt.show()


## 对比实验

下面先定义一个统一的训练入口，后续所有实验都直接调用这个函数。


In [ ]:
import pandas as pd
from IPython.display import display
from sklearn.metrics import confusion_matrix

from cnn import CNN, evaluate_model, train_model as fit_model

EPOCHS = 8
LR = 1e-3
criterion = torch.nn.CrossEntropyLoss()

def train_model(name, num_epochs=EPOCHS, lr=LR, **model_kwargs):
    model = CNN(**model_kwargs).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = fit_model(
        model=model,
        optimizer=optimizer,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        num_epochs=num_epochs,
        device=device,
        show_progress=True,
    )
    return {
        "name": name,
        "model": model,
        "config": model_kwargs,
        "lr": lr,
        "num_epochs": num_epochs,
        "params": sum(p.numel() for p in model.parameters()),
        **history,
    }

def summary_table(results):
    rows = []
    for name, result in results.items():
        rows.append({
            "模型": name,
            "参数量": result["params"],
            "最佳 epoch": result["best_epoch"],
            "最佳验证准确率": round(result["best_val_accuracy"], 4),
            "最佳验证损失": round(result["best_val_loss"], 4),
        })
    df = pd.DataFrame(rows).sort_values("最佳验证准确率", ascending=False).reset_index(drop=True)
    display(df)
    return df

def plot_validation_curves(results, title, save_name=None):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for name, result in results.items():
        epochs = range(1, len(result["val_accuracies"]) + 1)
        axes[0].plot(epochs, result["val_accuracies"], marker="o", label=name)
        axes[1].plot(epochs, result["val_losses"], marker="o", label=name)

    axes[0].set_title(f"{title}: Validation Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].grid(True)
    axes[0].legend()

    axes[1].set_title(f"{title}: Validation Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].grid(True)
    axes[1].legend()

    plt.tight_layout()
    if save_name is not None:
        plt.savefig(FIG_DIR / save_name, dpi=300, bbox_inches="tight")
    plt.show()

def evaluate_on_test(result):
    metrics = evaluate_model(
        model=result["model"],
        criterion=criterion,
        dataloader=test_loader,
        device=device,
    )
    return metrics

def collect_predictions(model, dataloader):
    model.eval()
    all_images, all_labels, all_preds = [], [], []
    with torch.no_grad():
        for images, labels in dataloader:
            logits = model(images.to(device))
            preds = logits.argmax(dim=1).cpu()
            all_images.append(images.cpu())
            all_labels.append(labels.cpu())
            all_preds.append(preds)

    return (
        torch.cat(all_images),
        torch.cat(all_labels),
        torch.cat(all_preds),
    )

def plot_confusion(cm, save_name=None):
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title("Confusion Matrix on Test Set")
    fig.colorbar(im, ax=ax)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            color = "white" if cm[i, j] > cm.max() * 0.5 else "black"
            ax.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=8, color=color)

    plt.tight_layout()
    if save_name is not None:
        plt.savefig(FIG_DIR / save_name, dpi=300, bbox_inches="tight")
    plt.show()

def show_misclassified_examples(images, labels, preds, save_name=None, max_items=12):
    error_index = torch.nonzero(labels != preds).squeeze(1)[:max_items]
    fig, axes = plt.subplots(3, 4, figsize=(10, 8))

    for ax, idx in zip(axes.flat, error_index):
        image = images[idx] * std + mean
        ax.imshow(image.squeeze(), cmap="gray")
        ax.set_title(f"true={class_names[labels[idx]]}\npred={class_names[preds[idx]]}", fontsize=9)
        ax.axis("off")

    for ax in axes.flat[len(error_index):]:
        ax.axis("off")

    plt.tight_layout()
    if save_name is not None:
        plt.savefig(FIG_DIR / save_name, dpi=300, bbox_inches="tight")
    plt.show()

def top_confusions(cm, topk=5):
    pairs = []
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            if i != j and cm[i, j] > 0:
                pairs.append((cm[i, j], class_names[i], class_names[j]))
    pairs.sort(reverse=True)
    return pd.DataFrame(pairs[:topk], columns=["错分数", "真实类别", "预测类别"])


In [ ]:
depth_results = {
    "2层卷积": train_model(
        "2层卷积",
        hidden_layers=2,
        hidden_channels=[32, 64],
        apply_pooling=[True, True],
        kernel_size=3,
        pooling_strategy="max",
        use_batch_norm=True,
        dropout=0.1,
    ),
    "3层卷积": train_model(
        "3层卷积",
        hidden_layers=3,
        hidden_channels=[32, 64, 128],
        apply_pooling=[True, True, False],
        kernel_size=3,
        pooling_strategy="max",
        use_batch_norm=True,
        dropout=0.1,
    ),
    "4层卷积": train_model(
        "4层卷积",
        hidden_layers=4,
        hidden_channels=[32, 64, 128, 128],
        apply_pooling=[True, True, False, False],
        kernel_size=3,
        pooling_strategy="max",
        use_batch_norm=True,
        dropout=0.1,
    ),
}

depth_table = summary_table(depth_results)


In [ ]:
plot_validation_curves(depth_results, "Depth Ablation", save_name="depth_ablation.png")


In [ ]:
baseline_result = depth_results["3层卷积"]

kernel_results = {
    "3x3": baseline_result,
    "5x5": train_model(
        "5x5",
        hidden_layers=3,
        hidden_channels=[32, 64, 128],
        apply_pooling=[True, True, False],
        kernel_size=5,
        pooling_strategy="max",
        use_batch_norm=True,
        dropout=0.1,
    ),
    "7x7": train_model(
        "7x7",
        hidden_layers=3,
        hidden_channels=[32, 64, 128],
        apply_pooling=[True, True, False],
        kernel_size=7,
        pooling_strategy="max",
        use_batch_norm=True,
        dropout=0.1,
    ),
}

kernel_table = summary_table(kernel_results)


In [ ]:
plot_validation_curves(kernel_results, "Kernel Size Ablation", save_name="kernel_ablation.png")


In [ ]:
pooling_results = {
    "Max Pooling": baseline_result,
    "Average Pooling": train_model(
        "Average Pooling",
        hidden_layers=3,
        hidden_channels=[32, 64, 128],
        apply_pooling=[True, True, False],
        kernel_size=3,
        pooling_strategy="avg",
        use_batch_norm=True,
        dropout=0.1,
    ),
}

pooling_table = summary_table(pooling_results)


In [ ]:
plot_validation_curves(pooling_results, "Pooling Ablation", save_name="pooling_ablation.png")


In [ ]:
all_results = {
    "2层卷积": depth_results["2层卷积"],
    "3层卷积": baseline_result,
    "4层卷积": depth_results["4层卷积"],
    "5x5": kernel_results["5x5"],
    "7x7": kernel_results["7x7"],
    "Average Pooling": pooling_results["Average Pooling"],
}

final_name, final_result = max(
    all_results.items(),
    key=lambda item: item[1]["best_val_accuracy"],
)
final_metrics = evaluate_on_test(final_result)

print(f"最终选择的模型: {final_name}")
print(f"最佳验证准确率: {final_result['best_val_accuracy']:.4f}")
print(f"测试集准确率: {final_metrics['accuracy']:.4f}")
print(f"测试集损失: {final_metrics['loss']:.4f}")


In [ ]:
test_images, test_labels, test_preds = collect_predictions(final_result["model"], test_loader)
cm = confusion_matrix(test_labels.numpy(), test_preds.numpy())
plot_confusion(cm, save_name="confusion_matrix.png")

error_summary = top_confusions(cm)
display(error_summary)


In [ ]:
show_misclassified_examples(test_images, test_labels, test_preds, save_name="misclassified_examples.png")
